# Emissions and Temperature

### Imports and preprocessing

In [1]:
import sys
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path

ROOT_DIR = Path.cwd().parent
Tables_DIR = ROOT_DIR / "data/tables"

# helper that saves each graph + summary so src/make_pdf.py can build the PDF
sys.path.insert(0, str(ROOT_DIR / "src"))
from report import save_report

event_countriesdf = pd.read_csv(Tables_DIR / "event_countries.csv")
eventsdf          = pd.read_csv(Tables_DIR / "events.csv")
country_yearlydf  = pd.read_csv(Tables_DIR / "country_yearly.csv")
global_yearlydf   = pd.read_csv(Tables_DIR / "global_yearly.csv")
coveragedf        = pd.read_csv(Tables_DIR / "data_coverage.csv")

for name, d in [("country_yearly", country_yearlydf), ("global_yearly", global_yearlydf),
                ("events", eventsdf), ("event_countries", event_countriesdf)]:
    print(name, list(d.columns))

# ---------- column auto-detection (edit candidates if needed) ----------
def col(df, *names):
    for n in names:
        if n in df.columns:
            return n
    raise KeyError(f"None of {names} found in {list(df.columns)}")

C_COUNTRY = col(country_yearlydf, "country", "country_name", "entity")
C_ISO     = col(country_yearlydf, "iso_code", "iso3", "country_code", "iso")
C_YEAR    = col(country_yearlydf, "year")
C_CO2     = col(country_yearlydf, "co2", "co2_mt", "annual_co2")
C_PC      = col(country_yearlydf, "co2_per_capita", "co2_pc", "co2_percap")
C_PM      = col(country_yearlydf, "pm25", "pm2_5", "pm25_exposure")

G_YEAR = col(global_yearlydf, "year")
G_TEMP = col(global_yearlydf, "temp_anomaly", "temperature_anomaly", "temp_anom", "anomaly", "j_d", "temp")

E_ID   = col(eventsdf, "event_id", "id")
E_YEAR = col(eventsdf, "year", "event_year")
E_NAME = col(eventsdf, "event_name", "name", "title", "event")
EC_ID  = col(event_countriesdf, "event_id", "id")
EC_ISO = col(event_countriesdf, "iso_code", "iso3", "country_code", "iso")

# real countries only (drop OWID aggregates like World, Asia, EU...)
is_country = (country_yearlydf[C_ISO].notna()
              & ~country_yearlydf[C_ISO].astype(str).str.startswith("OWID"))
cdf = country_yearlydf[is_country].copy()

# ---------- chart style ----------
NAVY, BLUE, RED, DARK = "#2c3e5e", "#4a78a8", "#e05252", "#2b2b2b"
FILL, GRID = "rgba(74,120,168,0.10)", "#e6eaf2"

def style(fig, title, subtitle, xt, yt, h=450, ylim=None):
    fig.update_layout(
        title=dict(text=f"<b>{title}</b><br><sup>{subtitle}</sup>",
                   x=0.02, xanchor="left", font=dict(size=20, color=NAVY)),
        template="plotly_white", width=1090, height=h,
        margin=dict(l=75, r=45, t=95, b=95),
        font=dict(color=NAVY, size=13),
        plot_bgcolor="white", paper_bgcolor="white",
        legend=dict(orientation="h", x=0.5, xanchor="center", y=-0.25),
    )
    fig.update_xaxes(title=xt, gridcolor=GRID, zeroline=False)
    fig.update_yaxes(title=yt, gridcolor=GRID, zeroline=False, range=ylim)
    return fig

def dot(x, y, label, color, name, pos="top left", showlegend=True):
    return go.Scatter(x=[x], y=[y], mode="markers+text", text=[label],
                      textposition=pos, name=name, showlegend=showlegend,
                      marker=dict(size=11, color=color), textfont=dict(color=NAVY))

def world_series(value_col):
    """World total per year = sum over real countries."""
    return cdf.groupby(C_YEAR)[value_col].sum(min_count=1)

country_yearly ['iso_code', 'country', 'year', 'population', 'gdp', 'co2_mt', 'co2_per_capita', 'cumulative_co2', 'share_global_co2', 'coal_co2', 'oil_co2', 'gas_co2', 'cement_co2', 'flaring_co2', 'methane', 'nitrous_oxide', 'total_ghg', 'pm25_exposure', 'co2_yoy_pct', 'rank_total_co2', 'rank_per_capita', 'co2_per_gdp', 'gdp_change_10yr_pct', 'co2_change_10yr_pct', 'decoupled_flag']
global_yearly ['year', 'decade', 'is_complete_year', 'co2_ppm', 'co2_mlo_ppm', 'ch4_ppb', 'n2o_ppb', 'temp_anomaly', 'world_co2_mt', 'co2_mlo_yoy_change', 'co2_pct_index', 'ch4_pct_index', 'n2o_pct_index', 'temp_yoy_change', 'co2_mlo_detrended', 'temp_detrended']
events ['event_id', 'event_name', 'year', 'event_type', 'source_note']
event_countries ['event_id', 'iso_code', 'role']


In [2]:
TOPIC = "emissions_temperature"

### Q1. How strongly do emissions correlate with temperature?

In [3]:
co2_w = world_series(C_CO2) / 1000
t = global_yearlydf.set_index(G_YEAR)[G_TEMP]
d = pd.concat([co2_w.rename("co2"), t.rename("temp")], axis=1).dropna()
d["cum"] = d["co2"].cumsum()

r_annual = d["co2"].corr(d["temp"])
r_cum    = d["cum"].corr(d["temp"])
m, b = np.polyfit(d["cum"], d["temp"], 1)
xs = np.array([d["cum"].min(), d["cum"].max()])

# ---- graph ----
fig = go.Figure()
fig.add_trace(go.Scatter(x=d["cum"], y=d["temp"], mode="markers", name="Year",
                         marker=dict(size=8, color=d.index, colorscale="Blues", showscale=False,
                                     line=dict(width=0.5, color=BLUE))))
fig.add_trace(go.Scatter(x=xs, y=m * xs + b, mode="lines", name=f"Trend (r = {r_cum:.2f})",
                         line=dict(color=RED, width=3)))
for idx in (0, -1):
    fig.add_trace(dot(d["cum"].iloc[idx], d["temp"].iloc[idx], f"{d.index[idx]}", DARK, "x", showlegend=False))
style(fig, "Cumulative CO2 Emissions vs Global Temperature",
      f"r = {r_cum:.2f} (cumulative), r = {r_annual:.2f} (annual emissions)",
      "Cumulative CO2 emitted (Gt)", "Temperature anomaly (°C)")
fig.show()

# ---- summary ----
summary = f"""
Over {d.index.min()}-{d.index.max()}, temperature correlates strongly with cumulative emissions (r = {r_cum:.2f})
and also with annual emissions (r = {r_annual:.2f}). Each additional 1,000 Gt of CO2 is associated with about
{m * 1000:.2f} °C of warming. Both series trend upward, so correlation alone overstates certainty, but the
cumulative relationship matches the physical expectation that warming tracks total CO2 added.
""".strip()
print(summary)
save_report(TOPIC, "q1", "How strongly do emissions correlate with temperature?", fig, summary)

Over 1880-2024, temperature correlates strongly with cumulative emissions (r = 0.96)
and also with annual emissions (r = 0.93). Each additional 1,000 Gt of CO2 is associated with about
0.76 °C of warming. Both series trend upward, so correlation alone overstates certainty, but the
cumulative relationship matches the physical expectation that warming tracks total CO2 added.


Resorting to unclean kill browser.


[report] saved emissions_temperature/q1 (1 graph(s))


### Q2. Does temperature respond with a delay to CO2?

In [4]:
dd = d[["co2", "temp"]].diff().dropna()
dd["co2"] = dd["co2"].rolling(3, center=True).mean()      # smooth year-to-year noise
dd["temp"] = dd["temp"].rolling(3, center=True).mean()
dd = dd.dropna()

lags = list(range(0, 21))
cc = [dd["co2"].corr(dd["temp"].shift(-L)) for L in lags]
best = int(np.nanargmax(np.abs(cc)))

# ---- graph ----
fig = go.Figure(go.Bar(x=lags, y=cc, marker_color=BLUE, name="Correlation"))
fig.add_trace(dot(lags[best], cc[best], f"Peak: {lags[best]} yrs<br>r = {cc[best]:.2f}", RED, "Strongest lag",
                  pos="top center"))
style(fig, "Does Temperature Respond to CO2 With a Delay?",
      "Correlation of year-on-year change in CO2 vs change in temperature, shifted by lag",
      "Lag (years, temperature after CO2)", "Correlation (r)")
fig.show()

# ---- summary ----
verdict = ("This suggests temperature lags emissions changes." if lags[best] > 0 else
           "The strongest link is contemporaneous, so no clear delay shows up at annual resolution.")
summary = f"""
Using smoothed year-to-year changes, the strongest correlation appears at a lag of {lags[best]} years (r = {cc[best]:.2f});
at zero lag r = {cc[0]:.2f}. {verdict}
Annual temperature is noisy (El Nino, volcanoes), so treat the exact lag as indicative rather than precise.
""".strip()
print(summary)
save_report(TOPIC, "q2", "Does temperature respond with a delay to CO2?", fig, summary)

Using smoothed year-to-year changes, the strongest correlation appears at a lag of 18 years (r = 0.23);
at zero lag r = 0.09. This suggests temperature lags emissions changes.
Annual temperature is noisy (El Nino, volcanoes), so treat the exact lag as indicative rather than precise.


Resorting to unclean kill browser.


[report] saved emissions_temperature/q2 (1 graph(s))
